In [0]:
%sql
create schema if not exists apexlife.silver;

In [0]:
bronze_table = 'apexlife.bronze.hospitals_raw'
silver_table = 'apexlife.silver.dim_hospital'
checkpoint_path = "abfss://data@apexlife.dfs.core.windows.net/silver/dim_hospital/checkpoint/"

In [0]:
from pyspark.sql.functions import * 

In [0]:
%sql select * from apexlife.bronze.hospitals_raw

hospital_id,hospital_name,city,bed_count,_rescued_data
H001,Apollo Main Hospital,Chennai,850,null
H002,Fortis Healthcare Delhi,Delhi,600,null
H003,Kokilaben Dhirubhai Hospital,Mumbai,700,null
H004,Manipal Hospital,Bangalore,400,null
H005,Yashoda Hospital,Hyderabad,450,null


In [0]:
df = (
    spark.readStream.table('apexlife.bronze.hospitals_raw')
)

In [0]:
df = (
    df
    .dropDuplicates(['hospital_id'])
    .withColumn('load_timestamp', current_timestamp())
)

In [0]:
from delta.tables import DeltaTable

In [0]:
def merge_hospital_dim(batch_df , batch_id) :
    if not spark.catalog.tableExists(silver_table) :
        batch_df.write.format('delta').mode('overwrite').saveAsTable(silver_table)
        return 
    
    hospital_dim = DeltaTable.forName(spark , silver_table)
    
    (
    hospital_dim.alias('t')
        .merge(
            batch_df.alias('s') ,
            't.hospital_id  = s.hospital_id'
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

(
    df.writeStream
        .foreachBatch(merge_hospital_dim)
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)

In [0]:
%sql select * from apexlife.bronze.hospitals_raw

hospital_id,hospital_name,city,bed_count,_rescued_data
H001,Apollo Main Hospital,Chennai,850,null
H002,Fortis Healthcare Delhi,Delhi,600,null
H003,Kokilaben Dhirubhai Hospital,Mumbai,700,null
H004,Manipal Hospital,Bangalore,400,null
H005,Yashoda Hospital,Hyderabad,450,null
